# Xuất model fine-tune sang GGUF để chạy bằng Ollama

**Settings:** Accelerator = *None* (CPU, ~30 GB RAM là đủ), Internet = *On*,
Add Input = dataset chứa `best_adapter` (ví dụ `modelhuhuver1`).

Các bước: gộp LoRA vào Qwen2.5-7B-Instruct (bf16) → chuyển sang GGUF f16 bằng llama.cpp → lượng tử hóa Q4_K_M (~4.7 GB).
Chạy *Save & Run All*, xong thì tải `qwen2.5-7b-huhu-Q4_K_M.gguf` ở tab Output về cùng thư mục với `serving/Modelfile`.

In [1]:
# KHÔNG cài requirements của llama.cpp: nó đổi phiên bản transformers / huggingface-hub / protobuf có sẵn của Kaggle
# và làm hỏng `from transformers import ...` (ImportError: GenerationMixin). Dùng transformers có sẵn để gộp model.
!df -h /tmp /kaggle/working | cat
!free -g | cat
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q peft            # chỉ cài nếu Kaggle chưa có, không nâng cấp
import transformers, peft, torch
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.0T  1.1T  87% /
/dev/loop2       20G  120K   20G   1% /kaggle/working
               total        used        free      shared  buff/cache   available
Mem:              31           1          25           0           4          30
Swap:              0           0           0
Cloning into 'llama.cpp'...
remote: Enumerating objects: 4132, done.
remote: Counting objects: 100% (4132/4132), done.
remote: Compressing objects: 100% (3426/3426), done.
remote: Total 4132 (delta 728), reused 2873 (delta 624), pack-reused 0 (from 0)
Receiving objects: 100% (4132/4132), 36.65 MiB | 17.08 MiB/s, done.
Resolving deltas: 100% (728/728), done.
Note: you may need to restart the kernel to use updated packages.
transformers 5.16.1 | peft 0.20.0 | torch 2.11.0+cpu


In [2]:
import glob, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-7B-Instruct"
hits = glob.glob("/kaggle/input/**/best_adapter/adapter_model.safetensors", recursive=True) \
    or glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
assert hits, "Không thấy adapter_model.safetensors trong /kaggle/input - hãy Add Input dataset chứa best_adapter"
ADAPTER = os.path.dirname(hits[0])
print("Adapter:", ADAPTER)

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
model.save_pretrained("/tmp/merged", safe_serialization=True, max_shard_size="4GB")
AutoTokenizer.from_pretrained(BASE).save_pretrained("/tmp/merged")
del model, base
print(os.listdir("/tmp/merged"))

Adapter: /kaggle/input/datasets/ghoenix/modelhuhuver1/best_adapter


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# Chỉ cài gói gguf của llama.cpp, không kéo theo phụ thuộc (giữ nguyên torch / transformers)
%pip install -q --no-deps ./llama.cpp/gguf-py
%pip install -q sentencepiece
!python llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/huhu-f16.gguf --outtype f16
!ls -lh /tmp/huhu-f16.gguf

In [ ]:
!rm -rf /tmp/merged
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize /tmp/huhu-f16.gguf /kaggle/working/qwen2.5-7b-huhu-Q4_K_M.gguf Q4_K_M
!rm -f /tmp/huhu-f16.gguf
!ls -lh /kaggle/working